In [253]:
import pandas as pd
import numpy as np
import math

import matplotlib.pyplot as plt
import seaborn as sns

import pickle
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# 1. Разведовательный анализ. Базовый анализ 

In [254]:
tmp_data = pd.read_excel('data/dataset.xlsx')

In [255]:
go_data = tmp_data.copy()

In [256]:
go_data.head(6)

,Unnamed: 0,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,...,0,0,0,0,0,0,0,0,3,0
1,1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,...,0,0,0,0,0,0,0,0,3,0
2,2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,...,0,0,0,0,0,0,0,0,3,0
3,3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,...,0,0,0,0,0,0,0,0,4,0
4,4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,...,0,0,0,0,0,0,0,0,0,0
5,5,15.037911,30.075821,2.000000,5.758408,5.758408,0.278083,0.278083,0.711012,28.600000,...,0,0,0,0,0,0,0,0,0,0


In [257]:
go_data = go_data.replace(0, None)

In [258]:
# Считаем долю NaN для каждого столбца и выводим самые пустые
display(go_data.isnull().mean().sort_values(ascending=False).head(500))

NumRadicalElectrons    1.0
SMR_VSA8               1.0
fr_phos_acid           1.0
fr_lactam              1.0
fr_N_O                 1.0
                      ... 
MolWt                  0.0
SPS                    0.0
MolMR                  0.0
HeavyAtomCount         0.0
MolLogP                0.0
Length: 214, dtype: float64

In [259]:
def drop_none_cols(X, proc=0.20): # удалим гргафы где больше 20: это пропуски - нули
    missing_series = X.isnull().mean().sort_values(ascending=False)
    lo_col_in_proc = missing_series[missing_series > proc].index.tolist()
    
    # сохраняем для предсказаний пул удаленных граф 
    lo_col_in_proc.append('Unnamed: 0')
    pd.DataFrame(lo_col_in_proc, columns=['Deleted_Columns']).to_csv('data/removed_cols.csv', index=False)

    # удаляем 
    X.drop(columns=lo_col_in_proc, inplace = True)

    return X

In [260]:
go_data = drop_none_cols(go_data)

In [261]:
display(go_data.isnull().mean().sort_values(ascending=False).head(500))

EState_VSA8       0.193806
VSA_EState2       0.182817
PEOE_VSA9         0.181818
SMR_VSA10         0.172827
SlogP_VSA6        0.156843
                    ...   
LabuteASA         0.000000
Kappa3            0.000000
HeavyAtomCount    0.000000
MolLogP           0.000000
MolMR             0.000000
Length: 79, dtype: float64

In [262]:
go_data.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 79 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   IC50, mM             1001 non-null   float64
 1   CC50, mM             1001 non-null   float64
 2   SI                   1001 non-null   float64
 3   MaxAbsEStateIndex    1001 non-null   float64
 4   MaxEStateIndex       1001 non-null   float64
 5   MinAbsEStateIndex    1001 non-null   float64
 6   MinEStateIndex       1001 non-null   float64
 7   qed                  1001 non-null   float64
 8   SPS                  1001 non-null   float64
 9   MolWt                1001 non-null   float64
 10  HeavyAtomMolWt       1001 non-null   float64
 11  ExactMolWt           1001 non-null   float64
 12  NumValenceElectrons  1001 non-null   int64  
 13  MaxPartialCharge     998 non-null    float64
 14  MinPartialCharge     998 non-null    float64
 15  MaxAbsPartialCharge  998 non-null    f

In [263]:
def set_zero(X):

    X = X.fillna(0).infer_objects(copy=False)
        
    return X 

In [264]:
def set_median(X): # пустоты заменим медианой при заменен на 0 вылезают выбросы 
    missing_series = X.isnull().mean().sort_values(ascending=False)
    lo_col_in_proc = missing_series[missing_series > 0].index.tolist()
    # сохраняем для предсказаний графа-медиана 
    
    
    # меняем в цикле что бы собратьь медианы 
    ll_to_save = [ ]
    for col in lo_col_in_proc: 
        l_median = X[col].median()
        X[col] = X[col].fillna(l_median).infer_objects(copy=False)
        ll_to_save.append({'Column': col, 'Median': l_median})

    lo_to_save = pd.DataFrame(ll_to_save)
    lo_to_save.to_csv('data/col_median.csv', index=False)
    return X

In [265]:
go_data = set_median(go_data)

/var/folders/qj/qtwhwk7s5939qz6m404hrczm0000gn/T/ipykernel_6747/3918525432.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].fillna(l_median).infer_objects(copy=False)
/var/folders/qj/qtwhwk7s5939qz6m404hrczm0000gn/T/ipykernel_6747/3918525432.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].fillna(l_median).infer_objects(copy=False)
/var/folders/qj/qtwhwk7s5939qz6m404hrczm0000gn/T/ipykernel_6747/3918525432.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change 

In [266]:
go_scaler = RobustScaler() # + IQR 

def ShowHisPlotAllCol(X):

    l_numeric_cols = X.select_dtypes(include=['number']).columns

    for col in l_numeric_cols:
        fig, axes =  plt.subplots(1, 3, figsize=(21, 8))
        sns.histplot(X[col], color='green', bins=10, kde=False, ax=axes[0])
        axes[0].set_title('Исходные данные')

        try: 
            #sns.histplot(np.log1p(X[col]), color='green', bins=10, kde=False, ax=axes[1])
            #axes[1].set_title('Логарифм (log1p)')


            scaled_data = go_scaler.fit_transform(X[[col]]).flatten()
            sns.histplot(scaled_data, color='red', bins=10, kde=False, ax=axes[2])
            axes[2].set_title('RobustScaler')
            
        except Exception as e:
            print(f'Пропущен столбец {col} из-за ошибки: {e}')
        
        plt.show()

#ShowHisPlotAllCol(go_data)



def set_RobustScaler(X):
    l_numeric_cols = X.select_dtypes(include=['number']).columns
    for col in l_numeric_cols:
        try: 
            X[col] = go_scaler.fit_transform(X[[col]])
        except Exception as e:
            print(f'Пропущен столбец {col} из-за ошибки: {e}')

    return X

#go_data = set_RobustScaler(go_data)

In [267]:
def ShowCorrCols(data, cols):
    plt.figure(figsize=(50,70))

    corr_matrix = data[cols].corr(numeric_only=True)
    sns.heatmap(corr_matrix, cmap='Blues', center=0, annot=True)
    plt.title('Correlogram', fontsize=22)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.show()

In [268]:
go_data.info(show_counts=True, verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 79 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   IC50, mM             1001 non-null   float64
 1   CC50, mM             1001 non-null   float64
 2   SI                   1001 non-null   float64
 3   MaxAbsEStateIndex    1001 non-null   float64
 4   MaxEStateIndex       1001 non-null   float64
 5   MinAbsEStateIndex    1001 non-null   float64
 6   MinEStateIndex       1001 non-null   float64
 7   qed                  1001 non-null   float64
 8   SPS                  1001 non-null   float64
 9   MolWt                1001 non-null   float64
 10  HeavyAtomMolWt       1001 non-null   float64
 11  ExactMolWt           1001 non-null   float64
 12  NumValenceElectrons  1001 non-null   int64  
 13  MaxPartialCharge     1001 non-null   float64
 14  MinPartialCharge     1001 non-null   float64
 15  MaxAbsPartialCharge  1001 non-null   f

In [269]:
go_data.to_csv('data/findata-2.csv')